# claude-colab Server

Gives Claude Code (or any AI coding agent) access to this GPU.

## Setup
1. **Runtime → Change runtime type → T4 GPU**
2. In Cell 1, pick a **tunnel**:
   - `cloudflare` (default): no account needed
   - `ngrok`: use this if `*.trycloudflare.com` is blocked on your network. Needs a free [ngrok account](https://dashboard.ngrok.com). Add your authtoken in the 🔑 **Secrets** tab as `NGROK_AUTHTOKEN` and enable notebook access.
3. Run all cells
4. Copy the connection string printed at the bottom
5. Paste it in your terminal: `claude-colab connect cc://TOKEN:KEY@host`

## What this does
- Starts a Flask API with 5 endpoints (exec, python, upload, download, health)
- All payloads are E2E encrypted with Fernet (AES-128-CBC + HMAC)
- Exposed via a Cloudflare quick tunnel (free, no signup) or ngrok (free account)
- Bearer token auth on all endpoints

## Install locally
```bash
pip install claude-colab
```

In [ ]:
# Cell 1: Choose tunnel and install dependencies
#@markdown Use `ngrok` if `*.trycloudflare.com` is blocked on your network.
TUNNEL = "cloudflare"  #@param ["cloudflare", "ngrok"]
#@markdown ngrok authtoken fallback. Prefer the `NGROK_AUTHTOKEN` secret (🔑 tab) so the token isn't saved in the notebook.
NGROK_AUTHTOKEN_FALLBACK = ""  #@param {type:"string"}

import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'flask', 'cryptography'], check=True)

if TUNNEL == 'cloudflare':
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared',
    ], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
elif TUNNEL == 'ngrok':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ngrok'], check=True)
    NGROK_AUTHTOKEN = None
    try:
        from google.colab import userdata
        NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
    except Exception:
        pass  # secret missing or notebook access not granted
    NGROK_AUTHTOKEN = NGROK_AUTHTOKEN or NGROK_AUTHTOKEN_FALLBACK.strip() or None
    if not NGROK_AUTHTOKEN:
        raise RuntimeError(
            'TUNNEL is "ngrok" but no authtoken was found.\n'
            '1. Sign up at https://dashboard.ngrok.com and copy your authtoken.\n'
            '2. Open the Secrets tab (key icon, left sidebar), add NGROK_AUTHTOKEN,\n'
            '   and enable notebook access. Or paste it into NGROK_AUTHTOKEN_FALLBACK.\n'
            '3. Re-run this cell.'
        )
else:
    raise ValueError(f'Unknown TUNNEL: {TUNNEL!r}')

print(f'Dependencies installed. Tunnel: {TUNNEL}')

In [ ]:
# Cell 2: Start API server with E2E encryption
import base64
import io
import json
import os
import secrets
import signal
import subprocess
import sys
import threading
import time
import traceback

from cryptography.fernet import Fernet, InvalidToken
from flask import Flask, request, jsonify

VERSION = '0.2.1'
BEARER_TOKEN = secrets.token_hex(32)
FERNET_KEY = Fernet.generate_key()
START_TIME = time.time()

fernet = Fernet(FERNET_KEY)

# Check GPU
try:
    import torch
    GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'
    GPU_AVAILABLE = torch.cuda.is_available()
except ImportError:
    GPU_NAME = 'torch not installed'
    GPU_AVAILABLE = False

if not GPU_AVAILABLE:
    print('WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU')


def decrypt_request():
    """Decrypt request body."""
    return json.loads(fernet.decrypt(request.data).decode('utf-8'))


def encrypt_response(data):
    """Encrypt response body."""
    plaintext = json.dumps(data).encode('utf-8')
    return fernet.encrypt(plaintext)


def check_auth():
    """Verify bearer token."""
    auth = request.headers.get('Authorization', '')
    if auth != f'Bearer {BEARER_TOKEN}':
        return jsonify({'error': 'unauthorized'}), 401
    return None


app = Flask(__name__)


@app.before_request
def auth_middleware():
    return check_auth()


@app.route('/health', methods=['GET'])
def health():
    try:
        import torch
        vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
        vram_used = torch.cuda.memory_allocated(0) / 1024**3 if torch.cuda.is_available() else 0
    except (ImportError, RuntimeError):
        vram_total = 0
        vram_used = 0

    disk = os.statvfs('/')
    disk_free = f'{disk.f_bavail * disk.f_frsize / 1024**3:.1f}GB'

    return jsonify({
        'status': 'ok',
        'version': VERSION,
        'gpu_name': GPU_NAME,
        'vram_total': round(vram_total, 1),
        'vram_used': round(vram_used, 1),
        'disk_free': disk_free,
        'uptime': int(time.time() - START_TIME),
        'python_version': sys.version.split()[0],
        'installed_packages': subprocess.getoutput('pip list --format=columns 2>/dev/null | head -20'),
    })


@app.route('/exec', methods=['POST'])
def exec_cmd():
    data = decrypt_request()
    command = data['command']
    timeout = min(data.get('timeout', 300), 600)

    t0 = time.time()
    try:
        result = subprocess.run(
            command, shell=True, capture_output=True, text=True, timeout=timeout,
        )
        return encrypt_response({
            'stdout': result.stdout,
            'stderr': result.stderr,
            'exit_code': result.returncode,
            'duration': round(time.time() - t0, 2),
        })
    except subprocess.TimeoutExpired:
        return encrypt_response({
            'stdout': '',
            'stderr': f'Command timed out after {timeout}s',
            'exit_code': -1,
            'duration': round(time.time() - t0, 2),
        })


@app.route('/python', methods=['POST'])
def python_exec():
    data = decrypt_request()
    code = data['code']
    timeout = min(data.get('timeout', 300), 600)

    t0 = time.time()
    stdout_capture = io.StringIO()
    namespace = {'_result': None}

    try:
        old_stdout = sys.stdout
        sys.stdout = stdout_capture
        exec(code, namespace)
        sys.stdout = old_stdout

        return_value = namespace.get('_result')
        try:
            json.dumps(return_value)
        except (TypeError, ValueError):
            return_value = str(return_value) if return_value is not None else None

        return encrypt_response({
            'output': stdout_capture.getvalue(),
            'error': None,
            'return_value': return_value,
            'duration': round(time.time() - t0, 2),
        })
    except Exception as e:
        sys.stdout = old_stdout
        return encrypt_response({
            'output': stdout_capture.getvalue(),
            'error': traceback.format_exc(),
            'return_value': None,
            'duration': round(time.time() - t0, 2),
        })


@app.route('/upload', methods=['POST'])
def upload_file():
    data = decrypt_request()
    path = data['path']
    content = base64.b64decode(data['content_b64'])

    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'wb') as f:
        f.write(content)

    return encrypt_response({
        'ok': True,
        'path': path,
        'size_bytes': len(content),
    })


@app.route('/download', methods=['POST'])
def download_file():
    data = decrypt_request()
    path = data['path']

    if not os.path.exists(path):
        return encrypt_response({'error': f'File not found: {path}'}), 404

    with open(path, 'rb') as f:
        content = f.read()

    return encrypt_response({
        'content_b64': base64.b64encode(content).decode(),
        'path': path,
        'size_bytes': len(content),
    })


# Start server in background
threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, threaded=True),
    daemon=True
).start()

print(f'API server running on port 5000')
print(f'GPU: {GPU_NAME}')
print(f'Version: {VERSION}')

In [ ]:
# Cell 3: Expose via the chosen tunnel and print connection string
import re
import subprocess
import threading
import time
from collections import deque

TUNNEL_STARTUP_TIMEOUT = 60  # seconds

if TUNNEL == 'ngrok':
    import ngrok
    # Keep a global reference so the listener stays open.
    listener = ngrok.forward(5000, authtoken=NGROK_AUTHTOKEN)
    host = listener.url().removeprefix('https://')
else:
    process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
    )
    tunnel_url = None
    recent_logs = deque(maxlen=20)

    def drain_cloudflared_logs():
        """Read cloudflared logs forever so a full pipe never blocks it."""
        global tunnel_url
        for line in process.stderr:
            recent_logs.append(line.rstrip())
            match = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
            if match and tunnel_url is None:
                tunnel_url = match.group(1)

    threading.Thread(target=drain_cloudflared_logs, daemon=True).start()

    deadline = time.time() + TUNNEL_STARTUP_TIMEOUT
    while tunnel_url is None:
        if process.poll() is not None:
            raise RuntimeError('cloudflared exited early:\n' + '\n'.join(recent_logs))
        if time.time() > deadline:
            process.terminate()
            raise RuntimeError(
                f'No tunnel URL after {TUNNEL_STARTUP_TIMEOUT}s. Last cloudflared logs:\n'
                + '\n'.join(recent_logs)
            )
        time.sleep(0.5)
    host = tunnel_url.removeprefix('https://')

key_str = FERNET_KEY.decode()
connection_string = f'cc://{BEARER_TOKEN}:{key_str}@{host}'

print()
print('=' * 70)
print('  claude-colab server ready!')
print(f'  GPU: {GPU_NAME}')
print(f'  Tunnel: {TUNNEL}')
print()
print('  Run this in your terminal:')
print(f'  claude-colab connect {connection_string}')
print()
print('  Security: E2E encrypted (Fernet AES-128), bearer token auth')
print('  Session: max 12h, 90min idle timeout')
print('=' * 70)
print()

# Keep alive
while True:
    time.sleep(60)